In [0]:
from pyspark.sql import functions as F, Window as W

cat = "capstone6_telecom.telecom"

EXTREME_LATENCY_MS = 500                      # TC02: exactly 35 rows are above this
EVENT_STATUS = ["OPEN", "FAILED", "PENDING", "COMPLETED"]
CDC_STATUS   = EVENT_STATUS + ["CANCELLED", "REVERSED"]
SEC_TYPES    = ["CHECK", "ALERT", "REVIEW"]
OUTAGE_STATUS = "FAILED"                      # a reading counts as an outage when its CURRENT status is this

### Helper functions (just run this cell)

In [0]:
def failed_rules(*pairs):
    """pairs = (condition_meaning_the_rule_FAILED, rule_id). Returns an array of failed rule ids."""
    return F.filter(F.array(*[F.when(c, F.lit(r)) for c, r in pairs]), lambda x: x.isNotNull())

def not_in(col, values):
    """True when the value is null or not in the list."""
    return F.coalesce(~F.col(col).isin(values), F.lit(True))

def rejected_rows(df, record_type, key_col, raw_cols):
    """df needs: rule_id, action, source_file, ingest_ts, key_col and raw_cols."""
    rules = spark.table(f"{cat}.silver_dq_rules").select("rule_id", F.col("description").alias("reason"))
    return (df.join(F.broadcast(rules), "rule_id", "left")
              .select(F.lit(record_type).alias("record_type"),
                      F.col(key_col).cast("string").alias("record_key"),
                      "rule_id", "action", "reason", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts"),
                      F.to_json(F.struct(*raw_cols)).alias("raw_record"),
                      F.current_timestamp().alias("rejected_at")))

def write_rejected(df, record_type):
    """Idempotent: replaces only this record_type's rows, so re-runs never duplicate."""
    t = f"{cat}.silver_rejected_records"
    if spark.catalog.tableExists(t):
        (df.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"record_type = '{record_type}'").saveAsTable(t))
    else:
        df.write.format("delta").saveAsTable(t)

### Functions: validate and apply status CDC (just run this cell)

In [0]:
def validate_cdc(raw, events):
    """Bronze CDC rows -> (valid, rejected). events needs measurement_id and event_ts."""
    raw_cols = raw.columns
    c = (raw
         .withColumn("change_ts", F.expr("try_to_timestamp(change_timestamp)"))
         .withColumn("seq", F.expr("try_cast(source_sequence AS BIGINT)"))
         .withColumn("prev_c", F.upper(F.trim(F.col("previous_status"))))
         .withColumn("new_c", F.upper(F.trim(F.col("new_status"))))
         .join(events.select("measurement_id", F.col("event_ts").alias("evt_ts")), "measurement_id", "left"))

    w = W.partitionBy("measurement_id", "seq").orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    c = (c.withColumn("rn", F.row_number().over(w))
          .withColumn("is_dup", F.col("seq").isNotNull() & (F.col("rn") > 1))
          .withColumn("failed", failed_rules(
              (F.col("evt_ts").isNull(), "DQ11"),                                   # orphan change
              (not_in("new_c", CDC_STATUS), "DQ12"),                                # unknown status
              (F.col("change_ts").isNull() | (F.col("change_ts") < F.col("evt_ts")), "DQ13"),  # bad / impossible time
              (F.col("seq").isNull(), "DQ14"))))                                    # no sequence

    valid = (c.filter(~F.col("is_dup") & (F.size("failed") == 0))
              .select("measurement_id", F.col("seq").alias("source_sequence"), "change_ts",
                      F.col("prev_c").alias("previous_status"), F.col("new_c").alias("new_status"),
                      "change_reason", "source_system", "source_file",
                      F.col("ingest_ts").alias("bronze_ingest_ts")))

    rej_fail = (c.filter(~F.col("is_dup") & (F.size("failed") > 0))
                 .withColumn("rule_id", F.explode("failed")).withColumn("action", F.lit("REJECTED")))
    rej_dup = c.filter(F.col("is_dup")).withColumn("rule_id", F.lit("DQ15")).withColumn("action", F.lit("DEDUPED"))
    rejected = rejected_rows(rej_fail.unionByName(rej_dup), "CDC", "measurement_id", raw_cols)
    return valid, rejected

def create_cdc_tables(history, current):
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {history} (
        measurement_id STRING, source_sequence BIGINT, change_ts TIMESTAMP,
        previous_status STRING, new_status STRING, status_changed BOOLEAN,
        change_reason STRING, source_system STRING, source_file STRING,
        bronze_ingest_ts TIMESTAMP, loaded_at TIMESTAMP)""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {current} (
        measurement_id STRING, region_id STRING, tower_id STRING, event_ts TIMESTAMP,
        initial_status STRING, current_status STRING, last_sequence BIGINT,
        last_change_ts TIMESTAMP, last_change_reason STRING, updated_at TIMESTAMP)""")

def apply_cdc(valid, events, history, current):
    """events needs: measurement_id, region_id, tower_id, event_ts, network_status."""
    events.select("measurement_id", "region_id", "tower_id", "event_ts", "network_status") \
          .createOrReplaceTempView("_cdc_events")
    valid.createOrReplaceTempView("_cdc_valid")

    # 1) seed: every event gets a current-state row (only new ones are inserted)
    spark.sql(f"""
      MERGE INTO {current} t USING _cdc_events s ON t.measurement_id = s.measurement_id
      WHEN NOT MATCHED THEN INSERT (measurement_id, region_id, tower_id, event_ts, initial_status,
           current_status, last_sequence, last_change_ts, last_change_reason, updated_at)
      VALUES (s.measurement_id, s.region_id, s.tower_id, s.event_ts, s.network_status,
           s.network_status, 0, s.event_ts, 'INITIAL_EVENT', current_timestamp())""")

    # 2) history: every valid change, once. Replays and late rows are kept, never duplicated.
    spark.sql(f"""
      MERGE INTO {history} t
      USING (SELECT measurement_id, source_sequence, change_ts, previous_status, new_status,
                    previous_status <> new_status AS status_changed, change_reason, source_system,
                    source_file, bronze_ingest_ts, current_timestamp() AS loaded_at FROM _cdc_valid) s
      ON t.measurement_id = s.measurement_id AND t.source_sequence = s.source_sequence
      WHEN NOT MATCHED THEN INSERT (measurement_id, source_sequence, change_ts, previous_status, new_status,
           status_changed, change_reason, source_system, source_file, bronze_ingest_ts, loaded_at)
      VALUES (s.measurement_id, s.source_sequence, s.change_ts, s.previous_status, s.new_status,
           s.status_changed, s.change_reason, s.source_system, s.source_file, s.bronze_ingest_ts, s.loaded_at)""")

    # 3) current: only a HIGHER source_sequence may overwrite, so late rows never beat newer status
    spark.sql(f"""
      MERGE INTO {current} t
      USING (SELECT measurement_id, source_sequence, change_ts, new_status, change_reason FROM (
               SELECT *, ROW_NUMBER() OVER (PARTITION BY measurement_id ORDER BY source_sequence DESC) AS rn
               FROM _cdc_valid) WHERE rn = 1) s
      ON t.measurement_id = s.measurement_id
      WHEN MATCHED AND s.source_sequence > t.last_sequence THEN UPDATE SET
           t.current_status = s.new_status, t.last_sequence = s.source_sequence,
           t.last_change_ts = s.change_ts, t.last_change_reason = s.change_reason,
           t.updated_at = current_timestamp()""")

In [0]:
events = spark.table(f"{cat}.silver_network_events")
raw    = spark.table(f"{cat}.bronze_status_updates")

valid, rejected = validate_cdc(raw, events)
write_rejected(rejected, "CDC")

In [0]:
create_cdc_tables(f"{cat}.silver_status_history", f"{cat}.silver_network_current")
apply_cdc(valid, events, f"{cat}.silver_status_history", f"{cat}.silver_network_current")

## Checks (run twice: the numbers must not change)

In [0]:
display(spark.sql(f"""
SELECT (SELECT COUNT(*) FROM {cat}.bronze_status_updates) AS bronze_cdc_rows,            -- 520
       (SELECT COUNT(*) FROM {cat}.silver_status_history) AS history_rows,               -- 520
       (SELECT COUNT(*) FROM {cat}.silver_rejected_records WHERE record_type='CDC') AS rejected_cdc, -- 0
       (SELECT COUNT(*) FROM {cat}.silver_status_history WHERE NOT status_changed) AS unchanged_status, -- 42
       (SELECT COUNT(*) FROM {cat}.silver_network_current) AS current_rows               -- 11,500"""))

display(spark.sql(f"""SELECT current_status, COUNT(*) AS rows FROM {cat}.silver_network_current
GROUP BY 1 ORDER BY rows DESC"""))
# expect COMPLETED 2891, OPEN 2772, FAILED 2755, PENDING 2733, CANCELLED 176, REVERSED 173

## Walk one measurement_id: initial event -> change -> current (good for the demo)

In [0]:
display(spark.sql(f"""
SELECT c.measurement_id, c.initial_status, h.previous_status, h.new_status, h.change_reason,
       h.source_sequence, h.change_ts, c.current_status
FROM {cat}.silver_network_current c JOIN {cat}.silver_status_history h USING (measurement_id)
ORDER BY h.source_sequence LIMIT 10"""))